In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
X_train=pd.read_csv("../data/X_rf_train.csv")
X_test=pd.read_csv("../data/X_rf_test.csv")
y_train=pd.read_csv("../data/y_train.csv")
y_test=pd.read_csv("../data/y_test.csv")

In [3]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(4279, 20)
(1070, 20)
(4279, 1)
(1070, 1)


In [4]:
y_train=y_train.squeeze()
y_test=y_test.squeeze()

In [5]:
print(y_train.shape)
print(y_test.shape)

(4279,)
(1070,)


In [2]:
import xgboost as xgb

In [7]:
model = xgb.XGBRegressor(objective='reg:squarederror',n_estimators=100, random_state=42)

In [8]:
model.fit(X_train,y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [9]:
y_train_pred=model.predict(X_train)
y_pred=model.predict(X_test)

In [10]:
from sklearn.metrics import(r2_score,mean_absolute_error,mean_squared_error)
r2=r2_score(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
rmse = np.sqrt(mean_squared_error(y_test,y_pred))

In [11]:
train_r2 = r2_score(y_train,y_train_pred)
print("Train R2:", round(train_r2,4))

Train R2: 0.999


In [12]:
print("TEST METRICS ARE")
print("R2 :", round(r2,4))
print("MAE :", round(mae,4))
print("RMSE :", round(rmse,4))

TEST METRICS ARE
R2 : 0.9799
MAE : 0.0311
RMSE : 0.0813


In [13]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(model,X_train,y_train,cv=10,scoring="r2",n_jobs=-1)
print("Fold Scores:")
print(cv_scores)
print()
print("Mean CV R2:",round(cv_scores.mean(),4))
print("Std CV R2:",round(cv_scores.std(),4))

Fold Scores:
[0.982736   0.96831825 0.98434163 0.98517771 0.96687794 0.98263743
 0.98206371 0.98333739 0.98572613 0.98380139]

Mean CV R2: 0.9805
Std CV R2: 0.0065


In [28]:
pip install optuna


Note: you may need to restart the kernel to use updated packages.


In [35]:
import optuna
from xgboost import XGBRegressor

In [40]:
def objective(trial):

    xgb_model = XGBRegressor(

        objective="reg:squarederror",

        n_estimators=trial.suggest_int(
            "n_estimators",
            100,
            1000
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            3,
            10
        ),

        learning_rate=trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        subsample=trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        min_child_weight=trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        gamma=trial.suggest_float(
            "gamma",
            0,
            5
        ),

        random_state=42,

        n_jobs=-1
    )

    scores = cross_val_score(
        xgb_model,
        X_train,
        y_train,
        cv=10,
        scoring="r2",
        n_jobs=-1
    )

    return scores.mean()

In [41]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=50
)

[I 2026-06-11 16:25:16,971] A new study created in memory with name: no-name-27b62355-f0c5-414c-a345-58ea7a4c218e


[I 2026-06-11 16:25:23,511] Trial 0 finished with value: 0.863991212315484 and parameters: {'n_estimators': 462, 'max_depth': 6, 'learning_rate': 0.07510561464804204, 'subsample': 0.9879792665708218, 'colsample_bytree': 0.8965284543101149, 'min_child_weight': 6, 'gamma': 4.213045349970361}. Best is trial 0 with value: 0.863991212315484.
[I 2026-06-11 16:25:26,691] Trial 1 finished with value: 0.9287270571791681 and parameters: {'n_estimators': 513, 'max_depth': 8, 'learning_rate': 0.08510659614300083, 'subsample': 0.8383137974253241, 'colsample_bytree': 0.9022032203373171, 'min_child_weight': 9, 'gamma': 0.7260305937218475}. Best is trial 1 with value: 0.9287270571791681.
[I 2026-06-11 16:25:26,967] Trial 2 finished with value: 0.808290188904406 and parameters: {'n_estimators': 126, 'max_depth': 7, 'learning_rate': 0.01397296393087988, 'subsample': 0.8340078100274374, 'colsample_bytree': 0.7159756209188589, 'min_child_weight': 6, 'gamma': 2.0430001475046833}. Best is trial 1 with value

In [42]:
print("Best CV R2:")
print(study.best_value)

print()

print("Best Parameters:")
print(study.best_params)

Best CV R2:
0.9809816085552413

Best Parameters:
{'n_estimators': 337, 'max_depth': 4, 'learning_rate': 0.2988939627715043, 'subsample': 0.8629215284372397, 'colsample_bytree': 0.6833721303155229, 'min_child_weight': 2, 'gamma': 0.001608288196587743}


In [43]:
best_xgb = XGBRegressor(
    objective="reg:squarederror",
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

best_xgb.fit(
    X_train,
    y_train
)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.6833721303155229
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import

In [44]:
y_train_pred = best_xgb.predict(X_train)
y_test_pred = best_xgb.predict(X_test)

In [45]:
from sklearn.metrics import(r2_score,mean_absolute_error,mean_squared_error)
r2=r2_score(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
rmse = np.sqrt(mean_squared_error(y_test,y_pred))

In [46]:
train_r2 = r2_score(y_train,y_train_pred)
print("Train R2:", round(train_r2,4))

Train R2: 0.9972


In [47]:
print("TEST METRICS ARE")
print("R2 :", round(r2,4))
print("MAE :", round(mae,4))
print("RMSE :", round(rmse,4))

TEST METRICS ARE
R2 : 0.9799
MAE : 0.0311
RMSE : 0.0813


In [ ]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(model,X_train,y_train,cv=10,scoring="r2",n_jobs=-1)
print("Fold Scores:")
print(cv_scores)
print()
print("Mean CV R2:",round(cv_scores.mean(),4))
print("Std CV R2:",round(cv_scores.std(),4))

Fold Scores:
[0.982736   0.96831825 0.98434163 0.98517771 0.96687794 0.98263743
 0.98206371 0.98333739 0.98572613 0.98380139]

Mean CV R2: 0.9805
Std CV R2: 0.0065


: 

In [14]:
df=pd.read_csv("../data/DataWithDescriptors.csv")
df.columns

Index(['Unnamed: 0', 'IL NAME', 'CATION NAME', 'ANION NAME', 'Cation_SMILES',
       'Anion_SMILES', 'Temperature (0 C)', 'Pressure (bar)',
       'CO2 capacity (mol CO2/mol IL)', 'Cation_Family',
       ...
       'An_fr_sulfide', 'An_fr_sulfonamd', 'An_fr_sulfone',
       'An_fr_term_acetylene', 'An_fr_tetrazole', 'An_fr_thiazole',
       'An_fr_thiocyan', 'An_fr_thiophene', 'An_fr_unbrch_alkane',
       'An_fr_urea'],
      dtype='str', length=444)

In [15]:
features=pd.read_excel("../data/RF_Top20_Features.xlsx")
features

,Descriptor
0,Pressure (bar)
1,Temperature (0 C)
2,Cat_MolLogP
3,Cat_EState_VSA2
4,An_TPSA
5,Cat_VSA_EState3
6,An_AvgIpc
7,An_SlogP_VSA1
8,Cat_TPSA
9,Cat_EState_VSA9


In [16]:
rf_features = features.iloc[:,0].tolist()

print(rf_features)
print(len(rf_features))

['Pressure (bar)', 'Temperature (0 C)', 'Cat_MolLogP', 'Cat_EState_VSA2', 'An_TPSA', 'Cat_VSA_EState3', 'An_AvgIpc', 'An_SlogP_VSA1', 'Cat_TPSA', 'Cat_EState_VSA9', 'Cat_BCUT2D_LOGPLOW', 'Cat_NumHDonors', 'Cat_SMR_VSA5', 'Cat_SMR_VSA1', 'Cat_PEOE_VSA1', 'Cat_NHOHCount', 'Cat_EState_VSA8', 'Cat_Chi4v', 'An_BalabanJ', 'An_PEOE_VSA3']
20


In [17]:
ranged_df=df[(df["Temperature (0 C)"] >= 25) &(df["Temperature (0 C)"] <= 30) &(df["Pressure (bar)"] >= 0.5) &(df["Pressure (bar)"] <= 1.5)].copy()

In [18]:
ranged_df.describe()

,Unnamed: 0,Temperature (0 C),Pressure (bar),CO2 capacity (mol CO2/mol IL),Cat_MaxAbsEStateIndex,Cat_MaxEStateIndex,Cat_MinAbsEStateIndex,Cat_MinEStateIndex,Cat_qed,Cat_SPS,...,An_fr_sulfide,An_fr_sulfonamd,An_fr_sulfone,An_fr_term_acetylene,An_fr_tetrazole,An_fr_thiazole,An_fr_thiocyan,An_fr_thiophene,An_fr_unbrch_alkane,An_fr_urea
count,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,146.000000,...,146.000000,146.000000,146.0,146.0,146.0,146.0,146.0,146.0,146.000000,146.0
mean,2130.917808,26.805822,0.937501,0.233495,3.115569,3.115569,1.035509,0.733522,0.475782,11.631347,...,0.006849,0.506849,0.0,0.0,0.0,0.0,0.0,0.0,0.068493,0.0
std,1585.237254,2.370709,0.277405,0.389334,1.868038,1.868038,0.252641,1.289823,0.148699,3.756898,...,0.082761,0.872939,0.0,0.0,0.0,0.0,0.0,0.0,0.400991,0.0
min,14.000000,25.000000,0.500000,0.003412,2.125000,2.125000,0.122336,-7.856082,0.076609,9.428571,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
25%,374.250000,25.000000,0.751500,0.021946,2.211806,2.211806,1.056389,1.056389,0.461795,9.909091,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
50%,2182.500000,25.010000,0.999000,0.039506,2.365434,2.365434,1.150139,1.150139,0.541779,10.200000,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
75%,3458.750000,30.000000,1.000000,0.214690,3.674173,3.674173,1.155926,1.155926,0.578815,11.583333,...,0.000000,1.500000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
max,4971.000000,30.000000,1.500000,2.050000,13.495084,13.495084,1.364796,1.364796,0.758479,24.666667,...,1.000000,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,4.000000,0.0


In [23]:
ranged_df["XG_Pred"] = model.predict(ranged_df[rf_features])

In [24]:
print(
    "Unique ILs:",
    ranged_df["IL NAME"].nunique()
)

Unique ILs: 72


In [26]:
rf_ranking = (ranged_df.groupby("IL NAME")["XG_Pred"].mean().reset_index())

In [27]:
top5_rf = rf_ranking.sort_values(by="XG_Pred",ascending=False).head(10)

print(top5_rf)

          IL NAME   XG_Pred
68    [TETAH][Pz]  2.049091
67    [TETAH][Py]  1.701037
66    [TETAH][Im]  1.694576
1     [AmemI][Br]  1.677461
69    [TETAH][Tz]  1.520531
57  [P66614][Gly]  1.265872
56  [P66614][Ala]  1.075536
71  [aP4443][Ala]  0.982325
58  [P66614][Ile]  0.961553
62  [P66614][Sar]  0.918744


In [5]:
X_train=pd.read_csv("../data/X_rf_trainpKa.csv")
X_test=pd.read_csv("../data/X_rf_testpKa.csv")
y_train=pd.read_csv("../data/y_trainpKa.csv")
y_test=pd.read_csv("../data/y_testpKa.csv")
y_train=y_train.squeeze()
y_test=y_test.squeeze()
model = xgb.XGBRegressor(objective='reg:squarederror',n_estimators=100, random_state=42)
model.fit(X_train,y_train)
y_train_pred=model.predict(X_train)
y_pred=model.predict(X_test)
from sklearn.metrics import(r2_score,mean_absolute_error,mean_squared_error)
r2=r2_score(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
rmse = np.sqrt(mean_squared_error(y_test,y_pred))
train_r2 = r2_score(y_train,y_train_pred)
print("Train R2:", round(train_r2,4))
print("TEST METRICS ARE")
print("R2 :", round(r2,4))
print("MAE :", round(mae,4))
print("RMSE :", round(rmse,4))

Train R2: 0.999
TEST METRICS ARE
R2 : 0.9788
MAE : 0.0307
RMSE : 0.0835


In [4]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(model,X_train,y_train,cv=10,scoring="r2",n_jobs=-1)
print("Fold Scores:")
print(cv_scores)
print()
print("Mean CV R2:",round(cv_scores.mean(),4))
print("Std CV R2:",round(cv_scores.std(),4))

Fold Scores:
[0.98131917 0.96981636 0.98618047 0.98438231 0.97140248 0.96652447
 0.97536731 0.98066268 0.98576521 0.98348991]

Mean CV R2: 0.9785
Std CV R2: 0.0068
